# 🎬 Dailies Jr.: a harness that makes a 20-second AI video
### ISOM 260 · Professor demo · Same model, different harness

**One idea in, a 20-second video out.** Same image model both times:

| | What happens |
|---|---|
| 🎲 **Level 0: just ask** | four separate prompts, four separate frames. Watch the hero change faces. |
| 🎬 **The harness** | 🎬 a **Director** writes a shot list → 🧑 one **character sheet** goes into every shot → ✅ a **Continuity Checker** (rules + code) sends bad shots back → 🖼️ every frame is drawn **from the first frame** → ✂️ an **editor** in code cuts the MP4 |
| 🎥 **Bonus** | one shot turned into real motion with Veo (costs real money, optional) |

**Rules make it likely. Code makes it checked.** Same lesson as today's Checker, now in film.

⚠️ *Run with the professor's own Gemini key (Colab 🔑 Secrets → `GOOGLE_API_KEY`). The class pass doesn't include image or video models.*

In [ ]:
!pip install -q -U "google-genai>=1.30"

In [ ]:
# @title ▶️ Run me: connect and find the models (no typing needed)
from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import display, HTML, Video
from PIL import Image, ImageDraw, ImageFont
import json, re, io, time, base64, subprocess, os, textwrap

KEY = (userdata.get("GOOGLE_API_KEY") or "").strip()
if ":" in KEY:
    raise SystemExit("⛔ That's a class pass. This demo needs a personal Gemini key (image and video models aren't on the class pass).")
client = genai.Client(api_key=KEY)
TEXT = "gemini-2.5-flash"

names = [m.name.split("/")[-1] for m in client.models.list()]
def pick(must, prefer=()):
    cands = [n for n in names if all(w in n for w in must)]
    for p in prefer:
        for n in cands:
            if p in n: return n
    return cands[0] if cands else None
IMAGE = pick(["gemini", "image"], prefer=["2.5-flash-image", "flash-image", "pro-image"])
VIDEO = pick(["veo"], prefer=["fast", "veo-3"])
print(f"✅ Connected. Text: {TEXT} · Image: {IMAGE or '❌ none found'} · Video: {VIDEO or '— none (bonus cell will skip)'}")
if not IMAGE: print("⚠️ No Gemini image model on this key. Pick OpenAI in the next cell for the storyboards.")

W, H = 1280, 720
os.makedirs("frames", exist_ok=True)
SPEND = {"images": 0, "text": 0, "video_s": 0}

def show_row(imgs, captions, title):
    tiles = ""
    for im, cap in zip(imgs, captions):
        b = io.BytesIO(); im.convert("RGB").resize((320, 180)).save(b, "JPEG", quality=85)
        tiles += (f"<div style='text-align:center;font:12px Arial;color:#333'><img style='border-radius:6px' src='data:image/jpeg;base64,{base64.b64encode(b.getvalue()).decode()}'>"
                  f"<div style='max-width:320px'>{cap}</div></div>")
    display(HTML(f"<div style='font:600 15px Arial;margin:8px 0'>{title}</div><div style='display:flex;gap:8px;flex-wrap:wrap'>{tiles}</div>"))

def fit(im):
    """Crop to 16:9 and resize, so every frame is the same size."""
    im = im.convert("RGB"); w, h = im.size; t = 16 / 9
    if w / h > t: nw = int(h * t); im = im.crop(((w - nw) // 2, 0, (w - nw) // 2 + nw, h))
    else: nh = int(w / t); im = im.crop((0, (h - nh) // 2, w, (h - nh) // 2 + nh))
    return im.resize((W, H))

def draw(prompt, ref=None, tries=3):
    """One image from the image model. ref = a PIL image the new frame must stay consistent with."""
    contents = [prompt] + ([ref] if ref is not None else [])
    for a in range(tries):
        try:
            r = client.models.generate_content(model=IMAGE, contents=contents,
                                               config=types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]))
            SPEND["images"] += 1
            for p in r.candidates[0].content.parts:
                if getattr(p, "inline_data", None) and p.inline_data.data:
                    return fit(Image.open(io.BytesIO(p.inline_data.data)))
            print("   (no image came back, asking again…)")
        except Exception as e:
            print(f"   (image model said {str(e)[:120]}… retrying)"); time.sleep(6 * (a + 1))
    raise SystemExit("⛔ The image model didn't return a picture. Run the cell again.")

def ask_json(prompt):
    for _ in range(3):
        r = client.models.generate_content(model=TEXT, contents=prompt,
              config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.4))
        SPEND["text"] += 1
        try: return json.loads(r.text)
        except Exception: print("   (not valid JSON, asking again…)")
    raise SystemExit("⛔ The Director didn't return valid JSON. Run the cell again.")

## 🎨 Pick the storyboard engine
Same harness, swap the artist. **Gemini** (default) or **OpenAI** (needs a Colab secret **`OPENAI_API_KEY`**). Shots 2–4 are drawn *from shot 1* either way, so the character stays the same. The storyboard then feeds the video (the cut, and AtlasCloud motion).

In [ ]:
# @title 🎨 Storyboard engine: Gemini or OpenAI (pick, then run)
# OpenAI needs a Colab secret named OPENAI_API_KEY (Notebook access on). The Director still thinks with Gemini.
STORYBOARD_ENGINE = "Gemini"  # @param ["Gemini", "OpenAI"]
OPENAI_QUALITY = "medium"  # @param ["low", "medium", "high"]
import requests

if STORYBOARD_ENGINE == "OpenAI":
    OAI_KEY = (userdata.get("OPENAI_API_KEY") or "").strip()
    if not OAI_KEY:
        raise SystemExit("⛔ Add a Colab secret OPENAI_API_KEY (🔑 left sidebar) and switch Notebook access on.")
    OAI_H = {"Authorization": f"Bearer {OAI_KEY}"}
    _r = requests.get("https://api.openai.com/v1/models", headers=OAI_H, timeout=30)   # free check
    if _r.status_code == 401: raise SystemExit("⛔ OpenAI says that key is wrong. Copy it again into OPENAI_API_KEY.")
    _have = {m["id"] for m in _r.json().get("data", [])} if _r.status_code == 200 else set()
    OAI_MODELS = [m for m in ["gpt-image-2", "gpt-image-1.5", "gpt-image-1"] if m in _have] or ["gpt-image-1"]
    IMAGE = OAI_MODELS[0]

    def _png(im):
        b = io.BytesIO(); im.convert("RGB").save(b, "PNG"); return b.getvalue()

    def _oai_call(model, prompt, ref, field):
        if ref is None:
            return requests.post("https://api.openai.com/v1/images/generations", headers=OAI_H, timeout=300,
                                 json={"model": model, "prompt": prompt, "size": "1536x1024", "quality": OPENAI_QUALITY, "n": 1})
        return requests.post("https://api.openai.com/v1/images/edits", headers=OAI_H, timeout=300,
                             data={"model": model, "prompt": prompt, "size": "1536x1024", "quality": OPENAI_QUALITY, "n": "1"},
                             files=[(field, ("shot1.png", _png(ref), "image/png"))])

    def draw(prompt, ref=None, tries=3):
        """One storyboard frame from OpenAI. ref = shot 1, so the character stays the same (the edits endpoint)."""
        global IMAGE
        field, a = "image[]", 0
        while a < tries:
            try:
                r = _oai_call(OAI_MODELS[0], prompt, ref, field)
            except Exception as e:
                a += 1; print(f"   (OpenAI: {str(e)[:100]}… retrying)"); time.sleep(5 * a); continue
            if r.status_code == 200:
                SPEND["images"] += 1
                d = r.json()["data"][0]
                raw = base64.b64decode(d["b64_json"]) if d.get("b64_json") else requests.get(d["url"], timeout=60).content
                return fit(Image.open(io.BytesIO(raw)))
            try: msg = str(r.json().get("error", {}).get("message", r.text))[:200]
            except Exception: msg = r.text[:200]
            low = msg.lower()
            if r.status_code == 401: raise SystemExit("⛔ OpenAI says that key is wrong.")
            if "billing" in low or "quota" in low: raise SystemExit(f"⛔ OpenAI billing/quota: {msg}")
            if "verif" in low or ("model" in low and r.status_code in (400, 403, 404) and "image" not in low.replace("gpt-image", "")):
                if len(OAI_MODELS) > 1:
                    print(f"   ({OAI_MODELS[0]} not available on this key, switching to {OAI_MODELS[1]})")
                    OAI_MODELS.pop(0); IMAGE = OAI_MODELS[0]; continue
            if ref is not None and field == "image[]" and r.status_code == 400 and "image" in low:
                field = "image"; continue
            a += 1; print(f"   (OpenAI said {msg[:120]}… retrying)"); time.sleep(5 * a)
        raise SystemExit("⛔ OpenAI didn't return a picture. Run the cell again.")

    print(f"🎨 Storyboards by OpenAI · {IMAGE} · quality {OPENAI_QUALITY} · shots 2–4 are edits of shot 1 (same character)")
else:
    print(f"🎨 Storyboards by Gemini · {IMAGE}")


## ✏️ The idea

In [ ]:
# ✏️ One line. Ask the room for one, or use this.
IDEA = "A 20-second ad for Rams Roast, a tiny coffee cart outside Suffolk University on a snowy Boston morning"

# 🎟️ PREMIERE: paste a team's ticket from the Studio notebook to render THEIR plan (their Director, their upgrades). Leave TODO to use IDEA.
TICKET = """TODO"""

TICKET_PLAN = None
if TICKET.strip() and not TICKET.strip().upper().startswith("TODO"):
    _t = TICKET.strip(); _j = json.loads(_t[_t.find("{"): _t.rfind("}") + 1])
    TICKET_PLAN, IDEA = _j["plan"], _j.get("idea", IDEA)
    print(f"🎟️ Premiere: team {_j.get('team')} · {_j.get('industry')} · upgrades: {', '.join(_j.get('upgrades', [])) or 'none'}")
    print(f"   Idea: {IDEA}")

## 🎲 Level 0: just ask (4 separate prompts)
No harness. Each frame is its own request, like four different people drawing the same story. **Bet first: will the main character look the same in all four?**

In [ ]:
# @title ▶️ Run me: Level 0
beats = ["opening shot", "the main character in action", "a close-up moment", "the final shot"]
L0 = [draw(f"Frame {i+1} of 4 of a short video. {IDEA}. This frame: {b}. Cinematic, 16:9.") for i, b in enumerate(beats)]
for i, im in enumerate(L0): im.save(f"frames/L0_{i+1}.png")
show_row(L0, beats, "🎲 Level 0: four prompts, no harness")
print("Same person in every frame? Same cart? Same weather? Hands up.")

## 🎬 The harness, step 1: the Director writes a shot list
The model doesn't draw yet. It **plans**: one character sheet, one look, four shots with timing, action, camera move and a caption. Planning before acting, like ReAct.

In [ ]:
# @title ▶️ Run me: the Director
PLAN = TICKET_PLAN or ask_json(f"""You are the director of a 20-second 16:9 social video. Idea: {IDEA}
Write a plan as ONLY JSON:
{{"title": "...",
  "character": {{"name": "...", "look": "ONE sentence, very specific and visual: age, hair, face, exact clothes and colors, one signature item"}},
  "style": "ONE sentence: art style, lighting, color palette, lens",
  "shots": [{{"n": 1, "seconds": 5, "action": "what we see, one sentence", "camera": "slow push in | slow pan left | slow pan right | static", "caption": "max 6 words"}}]}}
Exactly 4 shots. The character appears in every shot. No real brands, logos or real people.""")
card = f"<b>{PLAN['title']}</b><br>🧑 <b>{PLAN['character']['name']}</b>: {PLAN['character']['look']}<br>🎨 {PLAN['style']}<ol>" + \
       "".join(f"<li>[{s['seconds']}s · {s['camera']}] {s['action']} <i>“{s['caption']}”</i></li>" for s in PLAN["shots"]) + "</ol>"
display(HTML(f"<div style='font:14px Arial;border-left:5px solid #0a66c2;padding:8px 14px;max-width:760px'>{card}</div>"))

## ✅ Step 2: the Continuity Checker (rules + code)
Before a single picture is paid for, code checks the plan. Failures go back to the Director **once**, with the reasons. Every rejected shot here is money you don't spend on a bad render.

In [ ]:
# @title ▶️ Run me: check the plan in code
BANNED = ["logo", "starbucks", "dunkin", "nike", "apple", "taylor swift", "celebrity", "suffolk university logo"]
def check_plan(p):
    problems = []
    shots = p.get("shots", [])
    if len(shots) != 4: problems.append(f"needs exactly 4 shots, got {len(shots)}")
    total = sum(float(s.get("seconds", 0)) for s in shots)
    if not 16 <= total <= 24: problems.append(f"total length {total:g}s, must be 16–24s")
    for s in shots:
        if float(s.get("seconds", 0)) > 6: problems.append(f"shot {s.get('n')} is {s.get('seconds')}s, max 6")
        if len(str(s.get("caption", "")).split()) > 6: problems.append(f"shot {s.get('n')} caption is longer than 6 words")
        if s.get("camera") not in ("slow push in", "slow pan left", "slow pan right", "static"): problems.append(f"shot {s.get('n')} camera '{s.get('camera')}' isn't one our editor can do")
    text = json.dumps(p).lower()
    problems += [f"banned: '{w}'" for w in BANNED if w in text]
    if len(p.get("character", {}).get("look", "").split()) < 12: problems.append("character look is too vague (under 12 words): every shot depends on it")
    return problems

probs = check_plan(PLAN)
if probs:
    print("⛔ Checker FAIL:\n  - " + "\n  - ".join(probs) + "\n🔁 Back to the Director once, with the reasons…")
    PLAN = ask_json("Fix this video plan. Keep everything that is fine. Problems:\n- " + "\n- ".join(probs) + "\n\nPLAN:\n" + json.dumps(PLAN) + "\nReturn ONLY the fixed JSON, same format.")
    probs = check_plan(PLAN)
print("✅ PASS: the plan is safe to render." if not probs else "🟡 Still: " + "; ".join(probs) + " (we render anyway and say so in the memo)")

## 🖼️ Step 3: render, every frame drawn **from the first frame**
Shot 1 is drawn from the character sheet. Shots 2–4 get **shot 1 as a reference image** plus the same character sheet, word for word. That's the whole consistency trick: **the handoff is a document (and a picture), not a vibe.**

In [ ]:
# @title ▶️ Run me: render 4 keyframes (about a minute)
ch, style = PLAN["character"], PLAN["style"]
SHEET = f"Main character, {ch['name']}: {ch['look']}. Style: {style}. 16:9 cinematic frame, no text, no logos."
FRAMES = []
for s in PLAN["shots"]:
    prompt = f"{SHEET}\nShot {s['n']}: {s['action']}"
    if FRAMES:
        prompt = ("Keep the SAME character (same face, hair, clothes, signature item), the same place and the same style as the reference image. "
                  + prompt)
    im = draw(prompt, ref=FRAMES[0] if FRAMES else None)
    im.save(f"frames/H_{s['n']}.png"); FRAMES.append(im)
    print(f"   🖼️ shot {s['n']} done")
show_row(L0, ["Level 0"] * 4, "🎲 Level 0: no harness")
show_row(FRAMES, [s["caption"] for s in PLAN["shots"]], "🎬 Harness: same model, one character sheet, one reference frame")

## ✂️ Step 4: the editor is code, not a model
Camera moves, captions, crossfades and the MP4 are **deterministic code** (ffmpeg). Nothing to hallucinate, same result every time, and it costs $0.

In [ ]:
# @title ▶️ Run me: cut the video
def font(size):
    for f in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"]:
        if os.path.exists(f): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def captioned(im, text):
    im = im.copy(); d = ImageDraw.Draw(im, "RGBA"); f = font(54)
    w = d.textlength(text, font=f)
    d.rounded_rectangle([(W - w) / 2 - 28, H - 150, (W + w) / 2 + 28, H - 62], 18, fill=(0, 0, 0, 150))
    d.text(((W - w) / 2, H - 140), text, font=f, fill="white")
    return im

MOVES = {"slow push in": "z='min(zoom+0.0012,1.18)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)'",
         "slow pan left": "z=1.15:x='(iw-iw/zoom)*(1-on/{d})':y='ih/2-(ih/zoom/2)'",
         "slow pan right": "z=1.15:x='(iw-iw/zoom)*on/{d}':y='ih/2-(ih/zoom/2)'",
         "static": "z=1.0:x=0:y=0"}
FPS, clips = 25, []
for s, im in zip(PLAN["shots"], FRAMES):
    src, out = f"frames/C_{s['n']}.png", f"frames/clip_{s['n']}.mp4"
    captioned(im, s["caption"]).save(src)
    d = int(float(s["seconds"]) * FPS)
    move = MOVES.get(s["camera"], MOVES["static"]).replace("{d}", str(d))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", src, "-vf",
                    f"scale=2560:-1,zoompan={move}:d={d}:s={W}x{H}:fps={FPS},format=yuv420p",
                    "-t", str(float(s["seconds"])), "-c:v", "libx264", "-preset", "veryfast", out], check=True)
    clips.append(out)

# crossfade the clips together
cmd, filt, prev, offset = ["ffmpeg", "-y", "-loglevel", "error"], "", "0:v", 0.0
for c in clips: cmd += ["-i", c]
for i in range(1, len(clips)):
    offset += float(PLAN["shots"][i - 1]["seconds"]) - 0.5
    filt += f"[{prev}][{i}:v]xfade=transition=fade:duration=0.5:offset={offset:.2f}[v{i}];"
    prev = f"v{i}"
subprocess.run(cmd + ["-filter_complex", filt.rstrip(";"), "-map", f"[{prev}]", "-c:v", "libx264", "-preset", "veryfast",
                      "-pix_fmt", "yuv420p", "dailies_jr.mp4"], check=True)
print(f"🎬 dailies_jr.mp4 · {sum(float(s['seconds']) for s in PLAN['shots']) - 1.5:g}s")
display(Video("dailies_jr.mp4", embed=True, width=760))

## 🎥 Real motion with AtlasCloud: your 4 keyframes come alive
Same harness, one more model. Each keyframe goes to an AtlasCloud video model as the **first frame** of its shot, so the character stays the same. All shots render at once (1 to 5 minutes), and code cuts them together.

**Costs real money** (Seedance 2.0 Fast: about 7¢ per second, so about $1.15 for 4 shots × 4 s). Needs a Colab secret **`ATLASCLOUD_API_KEY`**. Tick the box, then run.

In [ ]:
# @title 🎥 AtlasCloud: bring the keyframes to life (real motion, costs real money)
# Needs a Colab secret named ATLASCLOUD_API_KEY (Notebook access on). Run "render 4 keyframes" first.
MAKE_ATLAS_VIDEO = False  # @param {type:"boolean"}
ATLAS_MODEL = "bytedance/seedance-2.0-fast/image-to-video"  # @param ["bytedance/seedance-2.0-fast/image-to-video", "kwaivgi/kling-v3.0-turbo/image-to-video", "google/veo3.1-lite/image-to-video", "alibaba/wan-2.6/image-to-video-flash"]
SECONDS_PER_SHOT = 4  # @param {type:"slider", min:4, max:8, step:1}
PRICE_PER_SEC = {"bytedance/seedance-2.0-fast/image-to-video": 0.072}  # others: see atlascloud.ai/pricing/models

import requests, os, concurrent.futures as cf
os.makedirs("frames", exist_ok=True)
ATLAS = "https://api.atlascloud.ai/api/v1/model"
n_shots = len(PLAN["shots"]) if "PLAN" in globals() else 4
est = PRICE_PER_SEC.get(ATLAS_MODEL)
print(f"🎥 {ATLAS_MODEL} · {n_shots} shots × {SECONDS_PER_SHOT}s"
      + (f" ≈ ${est * n_shots * SECONDS_PER_SHOT:.2f}" if est else " · price: see atlascloud.ai/pricing/models"))

def atlas_key():
    k = (userdata.get("ATLASCLOUD_API_KEY") or "").strip()
    if not k: raise SystemExit("⛔ Add a Colab secret ATLASCLOUD_API_KEY (🔑 left sidebar) and switch Notebook access on.")
    return k

def atlas_err(r):
    try: j = r.json()
    except Exception: return f"HTTP {r.status_code}: {r.text[:200]}"
    return f"HTTP {r.status_code}: {str(j.get('msg') or j.get('message') or j.get('error') or j)[:200]}"

def frame_data_uri(im):
    b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=88)
    return "data:image/jpeg;base64," + base64.b64encode(b.getvalue()).decode()

def upload_frame(im, key):
    """Fallback if the model won't take base64: upload the frame and get a URL back."""
    b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=88); b.seek(0)
    r = requests.post(f"{ATLAS}/uploadMedia", headers={"Authorization": f"Bearer {key}"},
                      files={"file": ("frame.jpg", b, "image/jpeg")}, timeout=60)
    if not r.ok: raise RuntimeError("upload failed: " + atlas_err(r))
    j = r.json(); d = j.get("data") if isinstance(j.get("data"), dict) else j
    url = d.get("url") or d.get("download_url") or (d.get("urls") or [None])[0]
    if not url: raise RuntimeError(f"upload returned no url: {str(j)[:200]}")
    return url

def submit(shot, im, key):
    prompt = (f"{shot['action']}. Camera: {shot.get('camera', 'static')}. Keep the same character, place and style as the first frame. "
              "Smooth, natural motion. No text, no logos.")
    body = {"model": ATLAS_MODEL, "prompt": prompt, "duration": int(SECONDS_PER_SHOT)}
    if "seedance" in ATLAS_MODEL: body.update({"resolution": "720p", "ratio": "16:9", "generate_audio": False})
    hdr = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    last = ""
    for how in ("base64", "upload"):
        img = frame_data_uri(im) if how == "base64" else upload_frame(im, key)
        for field in (("image",) if "seedance" in ATLAS_MODEL else ("image", "image_url")):
            r = requests.post(f"{ATLAS}/generateVideo", headers=hdr, json={**body, field: img}, timeout=120)
            if r.ok:
                j = r.json(); pid = (j.get("data") or {}).get("id")
                if pid: return pid
                last = f"no prediction id: {str(j)[:200]}"
            else:
                last = atlas_err(r)
                if r.status_code in (401, 402, 403): raise SystemExit(f"⛔ AtlasCloud said no ({last}). Check the key and your credit balance.")
    raise RuntimeError(last)

def wait(pid, key, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        r = requests.get(f"{ATLAS}/prediction/{pid}", headers={"Authorization": f"Bearer {key}"}, timeout=60)
        if r.ok:
            d = r.json().get("data") or {}
            st = str(d.get("status", "")).lower()
            if st in ("completed", "succeeded", "success"): return (d.get("outputs") or [None])[0]
            if st in ("failed", "error", "canceled", "cancelled"): raise RuntimeError(f"failed: {str(d.get('error'))[:200]}")
        time.sleep(5)
    raise RuntimeError("timed out after 10 minutes")

if not MAKE_ATLAS_VIDEO:
    print("Skipped. Tick MAKE_ATLAS_VIDEO and run again to spend money on real motion.")
elif "FRAMES" not in globals() or not FRAMES:
    raise SystemExit("⛔ Run 'render 4 keyframes' first: AtlasCloud animates those frames.")
else:
    key, t0 = atlas_key(), time.time()
    shots = PLAN["shots"]
    print(f"   sending {len(shots)} shots to AtlasCloud at once…")
    with cf.ThreadPoolExecutor(max_workers=len(shots)) as ex:
        pids = list(ex.map(lambda si: submit(si[0], si[1], key), zip(shots, FRAMES)))
    print("   ✅ all shots accepted. Rendering, usually 1 to 5 minutes…")
    def fetch(i_pid):
        i, pid = i_pid
        url = wait(pid, key)
        out = f"frames/atlas_{i + 1}.mp4"
        with open(out, "wb") as f: f.write(requests.get(url, timeout=180).content)
        print(f"   🎞️ shot {i + 1} done ({time.time() - t0:.0f}s)")
        return out
    with cf.ThreadPoolExecutor(max_workers=len(pids)) as ex:
        raw = list(ex.map(fetch, enumerate(pids)))
    # same size, same frame rate, then crossfade like the keyframe cut
    norm = []
    for i, c in enumerate(raw):
        o = f"frames/atlas_n{i + 1}.mp4"
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", c, "-an", "-vf",
                        f"scale={W}:{H}:force_original_aspect_ratio=increase,crop={W}:{H},fps=25,format=yuv420p",
                        "-c:v", "libx264", "-preset", "veryfast", o], check=True)
        norm.append(o)
    lens = [float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", c],
                                 capture_output=True, text=True).stdout.strip() or SECONDS_PER_SHOT) for c in norm]
    cmd, filt, prev, off = ["ffmpeg", "-y", "-loglevel", "error"], "", "0:v", 0.0
    for c in norm: cmd += ["-i", c]
    for i in range(1, len(norm)):
        off += lens[i - 1] - 0.5
        filt += f"[{prev}][{i}:v]xfade=transition=fade:duration=0.5:offset={off:.2f}[v{i}];"; prev = f"v{i}"
    final = "dailies_jr_motion.mp4"
    if len(norm) == 1: subprocess.run(["cp", norm[0], final], check=True)
    else: subprocess.run(cmd + ["-filter_complex", filt.rstrip(";"), "-map", f"[{prev}]", "-c:v", "libx264", "-preset", "veryfast",
                               "-pix_fmt", "yuv420p", final], check=True)
    secs = sum(lens) - 0.5 * (len(norm) - 1)
    print(f"🎬 {final} · {secs:.0f}s · {time.time() - t0:.0f}s to render"
          + (f" · about ${est * len(shots) * SECONDS_PER_SHOT:.2f}" if est else ""))
    display(Video(final, embed=True, width=760))


## 🎥 Bonus: real motion for one shot (Veo, costs real money)
Turns the harness's shot 1 frame into a real moving clip, starting from **that exact frame**, so continuity carries over. About 1–3 minutes, and roughly $0.10 per second of video on Veo Fast (about $0.80 for an 8-second clip) on a paid key. **Professor's key only.**

In [ ]:
# ✏️ Set to True to spend money on one real video clip.
MAKE_REAL_VIDEO = False

VEO_TRY = ([VIDEO] if VIDEO else []) + ["veo-3.1-fast-generate-preview", "veo-3.1-fast-generate-001", "veo-3.1-lite-generate-001",
                                        "veo-3.1-generate-preview", "veo-3.0-fast-generate-001"]
op = None
if not MAKE_REAL_VIDEO: print("Skipped. Set MAKE_REAL_VIDEO = True to try it.")
else:
    s, buf = PLAN["shots"][0], io.BytesIO(); FRAMES[0].save(buf, "PNG")
    for m in dict.fromkeys(VEO_TRY):
        try:
            op = client.models.generate_videos(model=m, prompt=f"{SHEET}\n{s['action']}. Camera: {s['camera']}. Gentle, realistic motion.",
                                               image=types.Image(image_bytes=buf.getvalue(), mime_type="image/png"),
                                               config=types.GenerateVideosConfig(aspect_ratio="16:9", number_of_videos=1))
            print(f"🎥 Using {m}"); break
        except Exception as e:
            print(f"   {m}: {str(e)[:160]}")
    if op is None:
        print("\n⛔ No Veo model accepted this key. Most likely the key's Google Cloud project has no billing: Veo is not on the free tier. "
              "In AI Studio → API keys, check the key's project shows a paid tier; turn on billing for it, then rerun.")
if op is not None:
    t0 = time.time()
    while not op.done:
        print(f"   🎥 rendering… {int(time.time() - t0)}s"); time.sleep(15); op = client.operations.get(op)
    v = op.response.generated_videos[0]
    client.files.download(file=v.video); v.video.save("veo_shot1.mp4")
    display(Video("veo_shot1.mp4", embed=True, width=760))

## 💵 The bill, and the lesson

In [ ]:
# @title ▶️ Run me
print(f"Image calls: {SPEND['images']} (4 for Level 0, 4 for the harness, plus any retries) · Director/fix calls: {SPEND['text']} · ffmpeg editor: $0")
print("\nWhat the harness added, and where each piece lives:")
for row in [("🎬 Director plans before drawing", "a model (one call)"),
            ("🧑 one character sheet, word for word, in every shot", "code (string)"),
            ("✅ Continuity Checker: length, camera, banned words, vague looks", "code (free, can't be talked around)"),
            ("🖼️ every shot drawn from shot 1", "a reference image (the handoff)"),
            ("✂️ camera moves, captions, crossfades, MP4", "code (ffmpeg, deterministic)")]:
    print(f"  {row[0]:<62} → {row[1]}")
print("\nSame model. Different harness. Which video would you pay for?")